# Turkish Lip-Reading — Full Pipeline Runner (Colab, GPU)

Runs **both pipelines** over `video_links.csv`, with all outputs written to
**Google Drive**. Whisper runs on the Colab **GPU** (set Runtime → Change runtime
type → GPU).

For each row, dispatched by the `pipeline` column:

| pipeline | what happens | outputs (on Drive) |
|---|---|---|
| **1** | download video → Whisper ASR | `raw_videos/`, `transcripts/<video>.json` |
| **2** | download → keep solo-reporter scenes → clip → Whisper per clip | `clips/<reporter>/`, `reporter_transcripts/`, appended to `reporter_clips.csv` |

**Resumable via the `completed` column.** When a row finishes it is set to
`completed = 1` and the CSV **is written back to Drive immediately**. Re-running
skips completed rows, so Colab timeouts are harmless — just run again.

### One-time setup in Google Drive
Create `MyDrive/turkish_lipreading/` containing:
```
turkish_lipreading/
├── video_links.csv              ← your latest CSV (with the `completed` column)
├── cookies.txt                  ← OPTIONAL but recommended (see Troubleshooting)
└── pipeline2_reporter_pictures/ ← reference photos, one <slug>.jpeg per reporter
    ├── fatih_portakal.jpeg
    ├── cuneyt_ozdemir.jpeg
    └── ...
```
Everything else (`raw_videos/`, `transcripts/`, `clips/`, `reporter_transcripts/`,
`reporter_clips.csv`) is created automatically.


## 1. Install dependencies
Same stack as local + Whisper. ~2 min.

In [ ]:
!pip install -q dlib-bin
!pip install -q --no-deps face_recognition
!pip install -q "git+https://github.com/ageitgey/face_recognition_models" 2>/dev/null
!pip install -q "setuptools<81"
!pip install -q "scenedetect[opencv]" imageio-ffmpeg
!pip install -q -U yt-dlp openai-whisper
!apt-get -qq install -y ffmpeg >/dev/null 2>&1
import torch
print("Torch CUDA available:", torch.cuda.is_available(),
      "| device:", (torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"))
print("If 'import face_recognition' fails below: Runtime > Restart session,")
print("then re-run from the Mount Drive cell (skip this install).")

## 2. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Config — edit these
Use the filters to chunk work (one reporter, or a few videos each) before a full run.

In [ ]:
from pathlib import Path

# ====== EDIT THESE ======
DRIVE_BASE = Path('/content/drive/MyDrive/turkish_lipreading')

WHISPER_MODEL = "large-v3"     # GPU makes this fast; use "small"/"base" for quick checks

# Limit the run (None = no limit)
PIPELINE_FILTER         = None     # None, "1", or "2"  -> run only that pipeline
REPORTERS_FILTER        = None     # e.g. ['fatih_portakal', 'cuneyt_ozdemir'] (pipeline 2)
MAX_ROWS                = None     # e.g. 5 to process only the first 5 pending rows

# Scene / face params (match local pipeline2.py)
SCENE_THRESHOLD = 30.0
SAMPLE_COUNT    = 5
FACE_TOLERANCE  = 0.5
MIN_DURATION    = 1.2
DOWNLOAD_FORMAT = "bestvideo[height>=720]+bestaudio/best/best"

# --- YouTube access (Colab IPs often hit the bot check; see Troubleshooting cell) ---
COOKIES_FILE     = DRIVE_BASE / 'cookies.txt'        # used automatically IF it exists
YT_PLAYER_CLIENT = "default,android,web_safari"      # extractor clients yt-dlp will try

KEEP_P2_SOURCE_VIDEO = False   # False = delete the big source video after clipping (saves Drive)
# ========================

CSV_PATH      = DRIVE_BASE / 'video_links.csv'
REF_DIR       = DRIVE_BASE / 'pipeline2_reporter_pictures'
RAW_DIR       = DRIVE_BASE / 'raw_videos'            # pipeline 1 full videos
TRANSCRIPTS   = DRIVE_BASE / 'transcripts'           # pipeline 1 transcripts
P2_RAW        = DRIVE_BASE / 'pipeline2_raw'         # pipeline 2 source videos (transient unless KEEP_P2_SOURCE_VIDEO)
CLIPS_DIR     = DRIVE_BASE / 'clips'                 # pipeline 2 clips
P2_TRANSCRIPTS= DRIVE_BASE / 'reporter_transcripts'  # pipeline 2 per-clip transcripts
REPORTER_CSV  = DRIVE_BASE / 'reporter_clips.csv'    # pipeline 2 manifest + transcript text

for d in (RAW_DIR, TRANSCRIPTS, P2_RAW, CLIPS_DIR, P2_TRANSCRIPTS):
    d.mkdir(parents=True, exist_ok=True)

print("CSV exists:", CSV_PATH.exists(), "->", CSV_PATH)
print("Reference dir exists:", REF_DIR.exists(), "->", REF_DIR)
print("cookies.txt present:", COOKIES_FILE.exists())
if REF_DIR.exists():
    print("Reference images:", sorted(p.name for p in REF_DIR.iterdir()))

## ⚠️ Troubleshooting: "download produced no file"
YouTube blocks most **Colab/datacenter IPs** with *"Sign in to confirm you're not a
bot"*, so `yt-dlp` returns no file. Two fixes, easiest first:

**A. Cookies (most reliable).** In your normal browser, install a "Get cookies.txt"
extension (e.g. *Get cookies.txt LOCALLY*), open `youtube.com` while logged in, export
`cookies.txt`, and drop it in `MyDrive/turkish_lipreading/`. The notebook uses it
automatically. Re-export if it stops working (cookies expire).

**B. Player clients.** `YT_PLAYER_CLIENT` in the config already tries several. If a
specific video still fails, run the **diagnostic cell below** to see the real error.

The download function now prints yt-dlp's actual error, so a failure tells you whether
it's the bot check, a removed video, or a format issue.

### Diagnostic (optional) — see the real yt-dlp error for one URL
Run this after the config + core-function cells if downloads fail.

In [ ]:
def diagnose(url):
    import subprocess, sys
    cmd = [sys.executable, "-m", "yt_dlp", "-F", "--no-playlist"]
    if COOKIES_FILE.exists():
        cmd += ["--cookies", str(COOKIES_FILE)]
    if YT_PLAYER_CLIENT:
        cmd += ["--extractor-args", f"youtube:player_client={YT_PLAYER_CLIENT}"]
    cmd += [url]
    print("Running:", " ".join(cmd), "\n")
    res = subprocess.run(cmd, capture_output=True, text=True)
    print(res.stdout)
    print(res.stderr)

# diagnose("https://www.youtube.com/watch?v=QgKPn7DZrgQ")  # uncomment to test

## 4. Core functions
Self-contained copies of the project's pipeline logic (download, scene detect,
solo-face filter, clip export) plus a Whisper transcriber that writes the same
JSON schema as `scripts/run_whisper.py`.

In [ ]:
import re, subprocess, sys, json, unicodedata
import cv2
import face_recognition
from scenedetect import SceneManager, open_video
from scenedetect.detectors import ContentDetector

def yt_dlp_base_cmd():
    return [sys.executable, "-m", "yt_dlp"]

def extract_video_id(url):
    for pat in (r"youtu\.be/([A-Za-z0-9_-]{11})",
                r"[?&]v=([A-Za-z0-9_-]{11})",
                r"/shorts/([A-Za-z0-9_-]{11})"):
        m = re.search(pat, url)
        if m:
            return m.group(1)
    return None

def normalize_text(text):
    return unicodedata.normalize("NFC", text)

def safe_name(raw):
    s = re.sub(r"[^A-Za-z0-9._-]+", "_", raw).strip("._")
    return s or "video"

def download_video(url, downloads_dir, start=None, end=None):
    downloads_dir.mkdir(parents=True, exist_ok=True)
    vid = extract_video_id(url) or "video"
    out_template = str(downloads_dir / "%(id)s.%(ext)s")
    cmd = [*yt_dlp_base_cmd(), "-f", DOWNLOAD_FORMAT,
           "--merge-output-format", "mp4", "--retries", "3",
           "--no-playlist", "-o", out_template]
    if COOKIES_FILE and Path(COOKIES_FILE).exists():
        cmd += ["--cookies", str(COOKIES_FILE)]
    if YT_PLAYER_CLIENT:
        cmd += ["--extractor-args", f"youtube:player_client={YT_PLAYER_CLIENT}"]
    if start and end:
        cmd += ["--download-sections", f"*{start}-{end}", "--force-keyframes-at-cuts"]
    cmd += [url]
    res = subprocess.run(cmd, capture_output=True, text=True)
    vids = [p for p in downloads_dir.glob(f"{vid}.*")
            if p.suffix.lower() in (".mp4", ".mkv", ".webm", ".mov")]
    if not vids:
        err = (res.stderr or res.stdout or "").strip().splitlines()
        tail = "\n      ".join(err[-6:]) if err else "(no yt-dlp output)"
        raise RuntimeError(f"download failed for {url}\n    yt-dlp said:\n      {tail}")
    return vids[0]

def load_reference_encoding(reference_image):
    image = face_recognition.load_image_file(str(reference_image))
    encs = face_recognition.face_encodings(image)
    if not encs:
        raise RuntimeError(f"No face found in reference image: {reference_image}")
    return encs[0]

def detect_scenes(video_path, threshold):
    video = open_video(str(video_path))
    mgr = SceneManager()
    mgr.add_detector(ContentDetector(threshold=threshold))
    mgr.detect_scenes(video=video, show_progress=False)
    scenes = []
    for s_tc, e_tc in mgr.get_scene_list():
        a, b = s_tc.get_seconds(), e_tc.get_seconds()
        if b > a:
            scenes.append((a, b))
    return scenes

def sample_times(start, end, n):
    dur = end - start
    if dur <= 0:
        return []
    if n <= 1:
        return [start + dur * 0.5]
    return [start + dur * ((i + 1) / (n + 1)) for i in range(n)]

def extract_frame_rgb(cap, t):
    cap.set(cv2.CAP_PROP_POS_MSEC, max(0, t * 1000))
    ok, frame = cap.read()
    if not ok or frame is None:
        return None
    return cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

def scene_has_only_target(cap, start, end, target_enc, n, tol):
    found = False
    for t in sample_times(start, end, n):
        rgb = extract_frame_rgb(cap, t)
        if rgb is None:
            continue
        locs = face_recognition.face_locations(rgb, model="hog")
        if len(locs) > 1:
            return False
        if len(locs) == 1:
            enc = face_recognition.face_encodings(rgb, known_face_locations=locs)
            if not enc:
                continue
            if face_recognition.compare_faces([target_enc], enc[0], tolerance=tol)[0]:
                found = True
    return found

def export_clip(input_video, output_video, start, end):
    output_video.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(["ffmpeg", "-hide_banner", "-loglevel", "error", "-y",
                    "-ss", f"{start:.3f}", "-to", f"{end:.3f}", "-i", str(input_video),
                    "-c:v", "libx264", "-preset", "veryfast", "-crf", "22",
                    "-c:a", "aac", str(output_video)], check=True)

# ---- Whisper (loaded once, on GPU if available) ----
_model = None
def get_model():
    global _model
    if _model is None:
        import whisper
        print(f"Loading Whisper '{WHISPER_MODEL}' ...")
        _model = whisper.load_model(WHISPER_MODEL)
    return _model

def transcribe(media_path, out_json):
    """Transcribe a video/clip; write run_whisper-style JSON; return plain text."""
    model = get_model()
    result = model.transcribe(str(media_path), word_timestamps=True,
                              language="tr", verbose=False)
    data = {"video": Path(media_path).name, "segments": []}
    texts = []
    for seg in result.get("segments", []):
        sd = {"start_sec": round(seg["start"], 3), "end_sec": round(seg["end"], 3),
              "text": normalize_text(seg["text"]), "words": []}
        texts.append(sd["text"])
        for w in seg.get("words", []):
            surface = normalize_text(w["word"])
            conf = round(w.get("probability", 1.0), 6)
            sd["words"].append({"surface_form": surface, "lemma": surface,
                                "start_sec": round(w["start"], 3), "end_sec": round(w["end"], 3),
                                "confidence": conf, "manual_review": int(conf < 0.7)})
        data["segments"].append(sd)
    out_json.parent.mkdir(parents=True, exist_ok=True)
    out_json.write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding="utf-8")
    return " ".join(texts).strip()

print("Core functions loaded.")

## 5. CSV + completion helpers
Reads `video_links.csv` from Drive, adds the `completed` column if missing, and
writes the whole CSV back to Drive each time a row finishes.

In [ ]:
import csv, os

LINK_FIELDS = ["url", "channel", "start", "end", "pipeline", "reporter", "completed"]
CLIP_FIELDS = ["reporter", "source_url", "source_video", "clip_file",
               "start_sec", "end_sec", "duration_sec",
               "transcript_text", "transcript_json"]

def col(row, key):
    return (row.get(key) or "").strip()

def load_rows():
    with open(CSV_PATH, newline="", encoding="utf-8") as f:
        rows = list(csv.DictReader(f))
    for r in rows:
        r.setdefault("completed", "")   # tolerate a CSV without the column yet
    return rows

def write_all_rows(rows):
    with open(CSV_PATH, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=LINK_FIELDS)
        w.writeheader()
        for r in rows:
            w.writerow({k: (r.get(k) or "") for k in LINK_FIELDS})

def mark_completed(rows, row):
    row["completed"] = "1"
    write_all_rows(rows)

def append_reporter_clips(clip_rows):
    if not clip_rows:
        return
    is_new = not REPORTER_CSV.exists()
    with open(REPORTER_CSV, "a", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=CLIP_FIELDS)
        if is_new:
            w.writeheader()
        for r in clip_rows:
            w.writerow(r)

def find_reference_image(reporter):
    for ext in (".jpeg", ".jpg", ".png", ".webp"):
        p = REF_DIR / f"{reporter}{ext}"
        if p.exists():
            return p
    return None

print("CSV/completion helpers loaded.")

## 6. Run
Processes pending (`completed != 1`) rows in CSV order. Outputs go to Drive; the
CSV is rewritten to Drive after every completed row.

In [ ]:
def process_pipeline1(row, rows):
    url, channel = col(row, "url"), col(row, "channel")
    start, end = col(row, "start"), col(row, "end")
    print(f"[P1] {channel} {url} {start}-{end if end else ''}")
    vpath = download_video(url, RAW_DIR, start or None, end or None)
    out_json = TRANSCRIPTS / (vpath.stem + ".json")
    if not out_json.exists():
        transcribe(vpath, out_json)
    mark_completed(rows, row)
    print(f"   -> transcript {out_json.name} (completed)")

_enc_cache = {}
def process_pipeline2(row, rows):
    reporter, url = col(row, "reporter"), col(row, "url")
    ref = find_reference_image(reporter)
    if not ref:
        print(f"[P2] skip {reporter}: no reference image in {REF_DIR}")
        return
    if reporter not in _enc_cache:
        _enc_cache[reporter] = load_reference_encoding(ref)
    enc = _enc_cache[reporter]
    print(f"[P2] {reporter} {url}")
    vpath = download_video(url, P2_RAW)
    scenes = detect_scenes(vpath, SCENE_THRESHOLD)
    cap = cv2.VideoCapture(str(vpath))
    base, clip_rows = safe_name(vpath.stem), []
    for idx, (s, e) in enumerate(scenes, start=1):
        if e - s < MIN_DURATION:
            continue
        if not scene_has_only_target(cap, s, e, enc, SAMPLE_COUNT, FACE_TOLERANCE):
            continue
        clip = CLIPS_DIR / reporter / f"{reporter}_{base}_sc{idx:04d}_{s:.2f}-{e:.2f}.mp4"
        export_clip(vpath, clip, s, e)
        txt = transcribe(clip, P2_TRANSCRIPTS / (clip.stem + ".json"))
        clip_rows.append({
            "reporter": reporter, "source_url": url, "source_video": vpath.name,
            "clip_file": str(clip.relative_to(DRIVE_BASE)),
            "start_sec": round(s, 3), "end_sec": round(e, 3),
            "duration_sec": round(e - s, 3),
            "transcript_text": txt,
            "transcript_json": str((P2_TRANSCRIPTS / (clip.stem + ".json")).relative_to(DRIVE_BASE)),
        })
    cap.release()
    if not KEEP_P2_SOURCE_VIDEO:
        try:
            vpath.unlink()
        except Exception:
            pass
    append_reporter_clips(clip_rows)
    mark_completed(rows, row)
    print(f"   -> {len(clip_rows)} clips ({len(scenes)} scenes) (completed)")

def run():
    rows = load_rows()
    pending = [r for r in rows if col(r, "completed") != "1"]
    if PIPELINE_FILTER:
        pending = [r for r in pending if col(r, "pipeline") == PIPELINE_FILTER]
    if REPORTERS_FILTER:
        pending = [r for r in pending
                   if col(r, "pipeline") != "2" or col(r, "reporter") in REPORTERS_FILTER]
    if MAX_ROWS:
        pending = pending[:MAX_ROWS]

    print(f"{len(pending)} pending rows to process\n")
    done = 0
    for row in pending:
        pl = col(row, "pipeline")
        try:
            if pl == "1":
                process_pipeline1(row, rows)
            elif pl == "2":
                process_pipeline2(row, rows)
            else:
                print(f"[skip] unknown pipeline '{pl}' for {col(row,'url')}")
                continue
            done += 1
        except Exception as e:
            print(f"   [FAIL] {col(row,'url')}: {e}")
    print(f"\nFinished. Completed {done}/{len(pending)} this run.")
    print(f"CSV updated on Drive: {CSV_PATH}")

run()

## 7. (Optional) Zip clips for download
Clips already live on Drive; this just bundles them.

In [ ]:
import shutil
archive = shutil.make_archive(str(DRIVE_BASE / "clips"), "zip", str(CLIPS_DIR))
print("Created:", archive)
# from google.colab import files; files.download(archive)